# Satellite-Guided Rainfall Gap Filling — GitHub + Colab

**This notebook pulls code from a GitHub repository; it does not embed Kemayoran observations.** The station CSV is uploaded privately at runtime. DEMO figures are **not scientific findings**.

Before starting: the GitHub repository is already published; have your separate `Kemayoran_Gauge_Private_Input.csv` ready.

## 1. Configure GitHub location
Paste the URL to **your** public repository. Do not paste tokens or passwords here.

In [ ]:
REPO_URL = "https://github.com/priambudihudaf/kemayoran-rainfall-gapfill.git"
GEE_PROJECT = ""  # Earth Engine-enabled Google Cloud project ID; not a password

assert "YOUR_USERNAME" not in REPO_URL, "Replace YOUR_USERNAME with your GitHub username first."

## 2. Clone latest code from GitHub and install Python packages
A fresh Colab session starts without any of your local files.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path('/content/kemayoran-rainfall-gapfill')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Repository path:', Path.cwd())

## 3. Upload the observation CSV into *private* runtime storage
Upload the **separate private file** provided with this project, not the original Excel. GitHub never receives it through this notebook.

In [ ]:
from google.colab import files
private_dir = Path('data/private')
private_dir.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Please upload exactly one normalized gauge CSV.')
file_name, csv_bytes = next(iter(uploaded.items()))
if not file_name.lower().endswith('.csv'):
    raise ValueError('A normalized .csv is required; see README for the separate input file.')
gauge_path = private_dir / 'kemayoran_gauge_daily_2020_2026.csv'
gauge_path.write_bytes(csv_bytes)
print('Private gauge saved to temporary Colab folder:', gauge_path)

## 4. Run offline checks and DEMO (synthetic satellite data only)

In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
subprocess.run([sys.executable,'src/run_analysis.py','--mode','demo',
                '--gauge-csv',str(gauge_path),'--repeats','10'],check=True)
print('DEMO only: plots in outputs/DEMO_SYNTHETIC_NOT_SCIENTIFIC')

## 5. Download genuine CHIRPS/IMERG via Earth Engine
You need your Google account to be registered with Earth Engine and the Cloud Project to be enabled. Authenticate interactively; do not put any credentials into GitHub. Verify daily observation-window dating before scientific interpretation.

In [ ]:
import ee
# Authenticate in the Colab *notebook* session, not inside a subprocess.
# A registered Earth Engine account + an enabled Cloud Project are required.
assert GEE_PROJECT.strip(), 'Fill GEE_PROJECT in step 1 first.'
ee.Authenticate(auth_mode='colab')
ee.Initialize(project=GEE_PROJECT)
print('Earth Engine notebook authentication successful.')
# The Python script reuses the saved credentials and only initializes EE.
subprocess.run([sys.executable, 'src/extract_satellite_gee.py',
                '--project', GEE_PROJECT, '--date-shift-days', '1',
                '--no-authenticate'], check=True)
print('Real satellite time series saved in data/private/')

## 6. Analyze the real satellite data (preliminary, time-alignment assumption must be checked)

In [ ]:
subprocess.run([sys.executable,'src/run_analysis.py','--mode','real',
                '--gauge-csv',str(gauge_path),'--repeats','100'],check=True)
print('Results: outputs/REAL_SATELLITE_RESULTS/')

## 7. Download results without committing private inputs to GitHub
Download, review QA, and decide separately what is legally and scientifically shareable.

In [ ]:
import shutil
from google.colab import files
results=Path('outputs/REAL_SATELLITE_RESULTS')
if not results.exists():
    raise FileNotFoundError('Run step 6 first.')
archive=shutil.make_archive('/content/Kemayoran_REAL_RESULTS_PRIVATE','zip',results)
files.download(archive)